# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: Trương Quốc Lê Huy

I have used the following AI tools: Gemini, ChatGPT

I understand that my submission needs to be my own work: Huy

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [2]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [3]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [4]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [5]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [6]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('east')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [7]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[2 1]'; Monster is at '[2 0]'
.....
.....
MH...
.....
.....
Hunter chose action 'north'
Monster chose action 'west'


Step 2: Hunter is at '[1 1]'; Monster is at '[2 0]'
.....
.H...
M....
.....
.....
Hunter chose action 'south'
Monster chose action 'stay'


Step 3: Hunter is at '[2 1]'; Monster is at '[2 0]'
.....
.....
MH...
.....
.....
Hunter chose action 'listen'
Monster chose action 'south'


Step 4: Hunter is at '[2 1]'; Monster is at '[3 0]'
.....
.....
.H...
M....
.....
Hunter chose action 'listen'
Monster chose action 'east'


Step 5: Hunter is at '[2 1]'; Monster is at '[3 1]'
.....
.....
.H...
.M...
.....
Hunter chose action 'teleport'
Monster chose action 'north'


Step 6: Hunter is at '[3 1]'; Monster is at '[2 1]'
.....
.....
.M...
.H...
.....
Hunter chose action 'south'
Monster chose action 'stay'


Step 7: Hunter is at '[4 1]'; Monster is at '[2 1]'
.....
.....
.M...
.....
.H...
Hunter chose action 'listen'
Monster chose action 'stay'


Step 8: Hunt

nan

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [8]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[50, nan, 25, nan, nan, 4, nan, 38, nan, 31, 90, 3, nan, 36, 77, 53, nan, 93, 63, nan, nan, 87, nan, nan, 40, nan, 2, nan, nan, 28, nan, 49, 4, 29, 0, nan, 6, 83, nan, nan, 3, 51, nan, nan, nan, nan, 1, nan, 9, 15, nan, 52, 27, 86, 44, 2, nan, nan, 17, 38, nan, 5, 93, nan, nan, nan, 14, 7, nan, 46, 93, 70, 13, 23, 46, nan, nan, nan, 51, nan, nan, nan, nan, nan, 91, 20, nan, nan, nan, nan, nan, nan, 77, 58, nan, 15, 5, nan, 95, nan]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [9]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 48 times.
Average steps to catch the monster over 52 successful runs: 39.58


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [12]:
import numpy as np

def model_based_hunter_agent_function(hunter_pos, monster_pos_instrument):
    # Khởi tạo trạng thái bộ nhớ lưu vị trí quái vật nếu chưa có
    if not hasattr(model_based_hunter_agent_function, "memory_pos"):
        model_based_hunter_agent_function.memory_pos = None

    # Nếu nhận được thông tin từ hành động listen trước đó, cập nhật bộ nhớ
    if monster_pos_instrument is not None:
        model_based_hunter_agent_function.memory_pos = monster_pos_instrument.copy()

    # Nếu chưa biết vị trí, hoặc đã đến vị trí trong trí nhớ mà không thấy quái vật -> lắng nghe
    if model_based_hunter_agent_function.memory_pos is None or np.array_equal(hunter_pos, model_based_hunter_agent_function.memory_pos):
        model_based_hunter_agent_function.memory_pos = None 
        return 'listen'

    # Có mục tiêu -> Rút ngắn khoảng cách Manhattan
    h_row, h_col = hunter_pos
    m_row, m_col = model_based_hunter_agent_function.memory_pos

    # Di chuyển ưu tiên trục dọc trước, sau đó đến trục ngang
    if h_row > m_row: 
        return 'north'
    elif h_row < m_row: 
        return 'south'
    elif h_col > m_col: 
        return 'west'
    elif h_col < m_col: 
        return 'east'

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [ ]:
sizes = [10, 30]
for n in sizes:
    # Chạy 100 lần, mỗi lần giới hạn số bước tối đa là n * 10
    steps = [arena_environment(model_based_hunter_agent_function, 
                               monster_agent_function_simple, 
                               n=n, max_steps=n*10, visualize=False) for _ in range(100)]
    
    steps = np.array(steps)
    fails = np.sum(np.isnan(steps))
    success_rate = (100 - fails) / 100 * 100
    avg_steps = np.nanmean(steps)
    
    print(f"--- KẾT QUẢ ĐẤU TRƯỜNG {n}x{n} ---")
    print(f"Tỷ lệ bắt thành công: {success_rate}%")
    if success_rate > 0:
        print(f"Số bước trung bình của các lần thành công: {avg_steps:.2f}\n")

--- KẾT QUẢ ĐẤU TRƯỜNG 10x10 ---
Tỷ lệ bắt thành công: 100.0%
Số bước trung bình của các lần thành công: 15.90

--- KẾT QUẢ ĐẤU TRƯỜNG 30x30 ---
Tỷ lệ bắt thành công: 100.0%
Số bước trung bình của các lần thành công: 41.01



## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

### 1. Chiến lược chọn hành động của thợ săn
Tác tử thợ săn sử dụng chiến lược **Model-based Reflex Agent** kết hợp thuật toán **Greedy** (Tham lam). 
Nó duy trì một bộ nhớ trạng thái (`memory_pos`) để ghi nhớ tọa độ mục tiêu thông qua hành động `listen`. Từ đó, nó chủ động di chuyển theo hướng làm giảm khoảng cách Manhattan lớn nhất thay vì đi ngẫu nhiên. Chiến lược này hoạt động cực kỳ hiệu quả vì nó loại bỏ hoàn toàn các bước đi vô định, tối ưu hóa lộ trình đến vị trí đã biết cuối cùng của quái vật.

### 2. Việc sử dụng Teleportation
**Không**, tôi không sử dụng `teleport` cho chiến lược ở môi trường này. 

**Lý do:** Trong đấu trường có kích thước vừa và nhỏ (10x10 hoặc 30x30), việc dịch chuyển tức thời mang rủi ro rất cao. Xác suất để `teleport` đưa tác tử đến gần quái vật thấp hơn nhiều so với xác suất bị đưa ra xa mục tiêu so với việc giữ nguyên vị trí và đi bộ áp sát.

### 3. Ảnh hưởng của kích thước đấu trường
Kích thước đấu trường càng lớn, **hiệu năng của tác tử càng giảm** (tỷ lệ bắt thành công giảm, số bước trung bình tăng). 
Không gian rộng làm tăng số bước cần thiết để tiếp cận mục tiêu. Trong thời gian thợ săn đi bộ, quái vật có nhiều không gian để tản bộ khỏi vị trí cũ hơn. Điều này khiến "bộ nhớ" của thợ săn nhanh chóng bị sai lệch, buộc thợ săn phải tốn thêm nhiều lượt `listen` để dò lại mục tiêu.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

### Cải tiến Agent Function của Quái vật
Để lẩn trốn thợ săn giỏi hơn, hàm tác tử của quái vật có thể được nâng cấp từ di chuyển ngẫu nhiên thành một **Tác tử phản xạ (Reflex Agent)**:
1. Tính toán khoảng cách Manhattan từ 4 ô liền kề xung quanh nó đến tọa độ hiện tại của thợ săn (giả sử quái vật cũng có cảm biến nhận biết vị trí thợ săn).
2. Lựa chọn di chuyển vào ô nào giúp tối đa hóa khoảng cách này.
3. Kết hợp logic kiểm tra biên (boundary check) để tránh tình trạng quái vật tự đi vào góc tường, mép bản đồ và bị kẹt đường lui.

# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.